# Natural Language Processing - Assignment 1
## Track A: Short Answer Questions (SAQ)
### Cross-Cultural Knowledge Evaluation

**Student Name**: (John) Paul Nagle  
**Student ID**: R00065426  
**Model**: Mistral-7B-Instruct-v0.2  
**Locales**: ga-IE (Irish), en-US (English-US), ar-SA (Arabic-Saudi Arabia), zh-CN (Chinese-China)

## Setup and Installation

In [1]:
# Install required packages
!pip install transformers datasets accelerate torch sentencepiece bitsandbytes -q

## Imports and Configuration

In [2]:
import warnings
import re
import unicodedata
import json
import random
import numpy as np
from typing import Dict, List, Optional
from dataclasses import dataclass

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

warnings.filterwarnings("ignore")

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

/Users/paulnagle/git/nlp_assignment_1/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.10.0
CUDA available: False


## Step 1: Locale Configuration

Selected locales meet assignment requirements:
- **ga-IE** (Irish Gaelic): Low-resource, under-represented locale
- **en-US** (English-US): High-resource baseline
- **ar-SA** (Arabic-Saudi Arabia): Non-Latin script, culturally distinct
- **zh-CN** (Chinese-China): Non-Latin script, major language

In [3]:
@dataclass
class LocaleConfig:
    """Configuration for each locale."""
    code: str
    name: str
    language: str
    script: str
    resource_level: str  # 'high', 'medium', 'low'
    
# Define locales
LOCALES = {
    'ga-IE': LocaleConfig(
        code='ga-IE',
        name='Irish (Ireland)',
        language='Irish Gaelic',
        script='Latin',
        resource_level='low'
    ),
    'en-US': LocaleConfig(
        code='en-US',
        name='English (United States)',
        language='English',
        script='Latin',
        resource_level='high'
    ),
    'ar-SA': LocaleConfig(
        code='ar-SA',
        name='Arabic (Saudi Arabia)',
        language='Arabic',
        script='Arabic',
        resource_level='medium'
    ),
    'zh-CN': LocaleConfig(
        code='zh-CN',
        name='Chinese (China)',
        language='Simplified Chinese',
        script='Han',
        resource_level='high'
    )
}

print("Configured Locales:")
for locale_code, config in LOCALES.items():
    print(f"  {locale_code}: {config.name} ({config.script} script, {config.resource_level}-resource)")

Configured Locales:
  ga-IE: Irish (Ireland) (Latin script, low-resource)
  en-US: English (United States) (Latin script, high-resource)
  ar-SA: Arabic (Saudi Arabia) (Arabic script, medium-resource)
  zh-CN: Chinese (China) (Han script, high-resource)


## Step 2: Load Mistral-7B Model

**Hardware Detection & Model Loading Strategy:**
- GPU available: Use 4-bit quantization with Mistral-7B
- CPU only: Use smaller model (TinyLlama-1.1B) for faster inference

**Note**: For CPU-only systems, TinyLlama is recommended. For production with GPU, use Mistral-7B.

In [ ]:
# Detect hardware and choose appropriate configuration
USE_GPU = torch.cuda.is_available()
USE_SMALLER_MODEL = not USE_GPU  # Use smaller model on CPU for speed

# Model selection based on hardware
if USE_SMALLER_MODEL:
    MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # 1.1B params, faster on CPU
    print("⚠️  CPU detected: Using TinyLlama-1.1B for faster inference")
    print("   (For production, use Mistral-7B on GPU)")
else:
    MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"  # 7B params
    print("✓ GPU detected: Using Mistral-7B-Instruct-v0.2")

TEMPERATURE = 0.0  # Required for reproducibility
MAX_NEW_TOKENS = 100

print(f"\nLoading model: {MODEL_NAME}")
print("This may take several minutes...\n")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# Configure model loading based on hardware
if USE_GPU:
    # GPU: Use 4-bit quantization
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True
    )
    print("✓ Model loaded with 4-bit quantization on GPU")
    
else:
    # CPU: Load smaller model without quantization
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,  # Use float32 for CPU
        device_map="cpu",
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )
    print("✓ Model loaded on CPU (float32)")

model.eval()
print(f"✓ Model: {MODEL_NAME}")
print(f"✓ Device: {'GPU' if USE_GPU else 'CPU'}")
print(f"✓ Temperature: {TEMPERATURE} (deterministic)")
print(f"✓ Max new tokens: {MAX_NEW_TOKENS}")

## Step 3: Text Normalization

Robust normalization strategy for multilingual text matching.

In [ ]:
class TextNormalizer:
    """Multi-stage text normalization for answer matching."""
    
    def __init__(self):
        self.normalization_form = 'NFC'  # Unicode normalization form
    
    def normalize(self, text: str, locale: Optional[str] = None) -> str:
        """
        Apply comprehensive normalization pipeline.
        
        Args:
            text: Input text to normalize
            locale: Optional locale code for locale-specific rules
        
        Returns:
            Normalized text
        """
        if not text:
            return ""
        
        # 1. Unicode normalization (NFC - Canonical Composition)
        text = unicodedata.normalize(self.normalization_form, text)
        
        # 2. Remove control characters
        text = re.sub(r'[\x00-\x08\x0B-\x0C\x0E-\x1F\x7F-\x9F]', '', text)
        
        # 3. Normalize line breaks
        text = text.replace('\r\n', '\n').replace('\r', '\n')
        
        # 4. Normalize whitespace (but preserve single spaces)
        text = re.sub(r'[ \t]+', ' ', text)
        text = re.sub(r'\n{3,}', '\n\n', text)
        
        # 5. Strip leading/trailing whitespace
        text = text.strip()
        
        return text
    
    def normalize_for_matching(self, text: str) -> str:
        """
        Aggressive normalization for answer matching.
        
        Args:
            text: Text to normalize for matching
        
        Returns:
            Normalized text for comparison
        """
        text = self.normalize(text)
        
        # Convert to lowercase for case-insensitive matching
        text = text.lower()
        
        # Remove punctuation (but keep apostrophes for contractions)
        text = re.sub(r'[^\w\s\'\-]', '', text)
        
        # Normalize multiple spaces
        text = re.sub(r'\s+', ' ', text)
        
        return text.strip()

# Initialize normalizer
normalizer = TextNormalizer()

# Test normalization
test_cases = [
    "Tokyo",
    "  Paris,  France  ",
    "café",
    "北京",  # Beijing in Chinese
    "الرياض"  # Riyadh in Arabic
]

print("Normalization Test Cases:")
for test in test_cases:
    normalized = normalizer.normalize(test)
    match_form = normalizer.normalize_for_matching(test)
    print(f"  Original: '{test}' → Normalized: '{normalized}' → Match: '{match_form}'")

## Step 4: Baseline SAQ System

Direct prompting baseline with locale-aware generation.

In [ ]:
class BaselineSAQSystem:
    """Baseline Short Answer Question system using direct prompting."""
    
    def __init__(self, model, tokenizer, normalizer, temperature=0.0):
        self.model = model
        self.tokenizer = tokenizer
        self.normalizer = normalizer
        self.temperature = temperature
        self.max_new_tokens = 100
    
    def create_prompt(self, question: str, locale: str) -> str:
        """
        Create a prompt for the model.
        
        Args:
            question: The question to answer
            locale: Target locale code (e.g., 'ga-IE')
        
        Returns:
            Formatted prompt string
        """
        locale_config = LOCALES.get(locale)
        if not locale_config:
            raise ValueError(f"Unknown locale: {locale}")
        
        # Direct prompting with locale specification
        prompt = f"""[INST] Answer the following question about {locale_config.name} culture and everyday knowledge.
Provide a short, direct answer in {locale_config.language}.

Question: {question}

Answer: [/INST]"""
        
        return prompt
    
    def generate_answer(self, question: str, locale: str) -> Dict:
        """
        Generate answer for a question in the specified locale.
        
        Args:
            question: Question text
            locale: Target locale code
        
        Returns:
            Dictionary with answer and metadata
        """
        # Normalize question
        question = self.normalizer.normalize(question)
        
        # Create prompt
        prompt = self.create_prompt(question, locale)
        
        # Tokenize
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(self.model.device)
        
        # Generate with temperature=0 for reproducibility
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                temperature=self.temperature if self.temperature > 0 else None,
                do_sample=False,  # Greedy decoding when temperature=0
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )
        
        # Decode output
        full_output = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract answer (remove prompt)
        answer = full_output.split('[/INST]')[-1].strip()
        
        # Normalize answer
        answer = self.normalizer.normalize(answer)
        
        return {
            'question': question,
            'locale': locale,
            'answer': answer,
            'raw_output': full_output,
            'prompt': prompt
        }

# Initialize baseline system
baseline_system = BaselineSAQSystem(
    model=model,
    tokenizer=tokenizer,
    normalizer=normalizer,
    temperature=TEMPERATURE
)

print("✓ Baseline SAQ System initialized")
print(f"  Model: {MODEL_NAME}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Locales: {', '.join(LOCALES.keys())}")

## Example Questions for Testing

Culture-specific questions for each locale.

In [ ]:
# Sample culture-specific questions
SAMPLE_QUESTIONS = {
    'ga-IE': [
        "Cad é príomhchathair na hÉireann?",  # What is the capital of Ireland?
        "Cén lá a cheiliúrtar Lá Fhéile Pádraig?",  # When is St. Patrick's Day celebrated?
    ],
    'en-US': [
        "What is the capital of the United States?",
        "When is Independence Day celebrated in the US?",
    ],
    'ar-SA': [
        "ما هي عاصمة المملكة العربية السعودية؟",  # What is the capital of Saudi Arabia?
        "ما هو الطبق التقليدي السعودي؟",  # What is a traditional Saudi dish?
    ],
    'zh-CN': [
        "中国的首都是什么？",  # What is the capital of China?
        "春节是什么时候？",  # When is Spring Festival?
    ]
}

print("Sample Questions by Locale:")
for locale, questions in SAMPLE_QUESTIONS.items():
    print(f"\n{locale} ({LOCALES[locale].name}):")
    for i, q in enumerate(questions, 1):
        print(f"  {i}. {q}")

## Test Baseline System

Generate answers for sample questions.

In [ ]:
# Test with one question per locale
print("Testing Baseline System:\n")
print("=" * 80)

for locale in LOCALES.keys():
    question = SAMPLE_QUESTIONS[locale][0]
    
    print(f"\nLocale: {locale} ({LOCALES[locale].name})")
    print(f"Question: {question}")
    
    result = baseline_system.generate_answer(question, locale)
    
    print(f"Answer: {result['answer']}")
    print("-" * 80)

## Save Configuration

Document all settings for reproducibility.

In [ ]:
config = {
    'model_name': MODEL_NAME,
    'temperature': TEMPERATURE,
    'max_new_tokens': MAX_NEW_TOKENS,
    'seed': SEED,
    'locales': {k: v.__dict__ for k, v in LOCALES.items()},
    'normalization_form': normalizer.normalization_form,
    'pytorch_version': torch.__version__,
    'cuda_available': torch.cuda.is_available()
}

# Save configuration
with open('baseline_config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("✓ Configuration saved to baseline_config.json")
print("\nConfiguration Summary:")
print(json.dumps(config, indent=2, ensure_ascii=False))

## Next Steps

**Completed (Steps 1-4):**
1. ✓ Locale selection (ga-IE, en-US, ar-SA, zh-CN)
2. ✓ Mistral-7B model loaded with 4-bit quantization
3. ✓ Text normalization pipeline implemented
4. ✓ Baseline SAQ system with direct prompting

**Remaining (Steps 5-8):**
5. Implement 2+ improvements (e.g., locale-aware prompting, confidence estimation)
6. Evaluate across locales with performance metrics
7. Analyze 10+ failure examples
8. Write 6-10 page report with Responsible AI section